In [1]:
"""
Project: Large-Scale Social Media Intelligence Platform
Tech Stack: Pandas, Scikit-Learn, BERTopic / LDA, NetworkX (Graph Analysis)
"""

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import networkx as nx

def run_social_media_pipeline():
    print("=== [DS-03] Running Social Media Intelligence Platform Pipeline ===")
    
    # Step 1: Simulated Raw Data (Reddit/YouTube API structure)
    data = {
        'post_id': [101, 102, 103, 104, 105],
        'username': ['user_alpha', 'trend_setter', 'bot_fake99', 'tech_guru', 'news_daily'],
        'followers_count': [1200, 45000, 15, 89000, 250000],
        'post_text': [
            "Loving the new AI tech trends coming up this year!",
            "Biggest discount sale of the season! Click here to buy.",
            "BREAKING: Secret miracle cure discovered that doctors hate!",
            "Python and Machine Learning are scaling new heights in data science.",
            "Weekly global news update on technology and innovation."
        ],
        'retweets': [12, 450, 2, 890, 2300]
    }
    df = pd.DataFrame(data)
    
    # Step 2: Text Cleaning & Sentiment Heuristics
    df['clean_text'] = df['post_text'].str.lower().replace(r'http\S+', '', regex=True)
    positive_words = ['loving', 'amazing', 'sale', 'scaling', 'innovation']
    df['sentiment'] = df['clean_text'].apply(lambda x: 'Positive' if any(w in x for w in positive_words) else 'Neutral')
    
    # Step 3: Misinformation Classifier Heuristic
    def detect_fake_news(row):
        text = row['post_text'].lower()
        if ('miracle' in text or 'doctors hate' in text or 'click here' in text) and row['followers_count'] < 100:
            return 1 # Fake / Misinformation
        return 0 # Real
    df['misinformation_label'] = df.apply(detect_fake_news, axis=1)
    
    # Step 4: Topic Modeling (LDA / Vectorization)
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(df['post_text'])
    lda = LatentDirichletAllocation(n_components=2, random_state=42)
    lda.fit(X)
    print(f"Topic Modeling completed. Extracted {lda.n_components} latent topics.")
    
    # Step 5: Graph Analysis & Influencer Ranking using NetworkX (Neo4j simulation)
    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_node(row['username'], followers=row['followers_count'])
    
    # Adding simulated interactions (edges)
    G.add_edge('user_alpha', 'tech_guru')
    G.add_edge('trend_setter', 'news_daily')
    G.add_edge('bot_fake99', 'user_alpha')
    G.add_edge('tech_guru', 'news_daily')
    
    # PageRank for Influencer ranking
    pagerank_scores = nx.pagerank(G)
    df['pagerank_score'] = df['username'].map(pagerank_scores)
    
    print("\nProcessed Social Media Intelligence DataFrame:")
    print(df[['username', 'sentiment', 'misinformation_label', 'pagerank_score']])
    return df

if __name__ == "__main__":
    run_social_media_pipeline()

=== [DS-03] Running Social Media Intelligence Platform Pipeline ===
Topic Modeling completed. Extracted 2 latent topics.

Processed Social Media Intelligence DataFrame:
       username sentiment  misinformation_label  pagerank_score
0    user_alpha  Positive                     0        0.176879
1  trend_setter  Positive                     0        0.095610
2    bot_fake99   Neutral                     1        0.095610
3     tech_guru  Positive                     0        0.245957
4    news_daily  Positive                     0        0.385944
